# 03.5 — Normalization recipes

Raw ephys signals have wildly different scales — channel to channel, session to session. Normalization brings them to a common range so the network sees comparable inputs. This project ports MATLAB's *string-driven* normalization dispatch: a recipe name like `"Channel - Z-Score - Global - MinMax - [-1,1] - Zero Centered - Range 0.5"` selects a composed transform.

It's also a working example of two things the curriculum keeps returning to: the **registry pattern** on real data, and the honesty of reading a **mid-migration codebase** — where some pieces are live and others are documented stubs.

This notebook covers:

* Why normalize, shown on real signal scales.
* The recipe-string grammar and the registry that dispatches it.
* What's actually implemented today (only two recipes) vs. stubbed for a later milestone — and how to tell.
* The stats-table requirement and its silent-passthrough trap.

**Prerequisites:** [01.3 functions](../01_python_for_matlab_users/01.3_functions_and_lambdas.ipynb) (functions as values / registry), [02.1 arrays](../02_numpy_and_pytorch_basics/02.1_numpy_vs_matlab_arrays.ipynb) (axis reductions, broadcasting).

## Section 1 — What MATLAB does

`cgg_selectNormalization.m` is a big `switch` over the recipe string, applying a precomputed statistics table:

```matlab
function Data = cgg_selectNormalization(Data, NormalizationTable, Normalization)
switch Normalization
    case 'Channel - Z-Score'
        Data = (Data - Mean_perChannel) ./ STD_perChannel;
    case 'Global - MinMax - [-1,1]'
        Data = 2 * (Data - Min_global) ./ (Max_global - Min_global) - 1;
    case 'None'
        % passthrough
end
```

Two design points to carry over: the **statistics** (mean, std, min, max per channel/area/global) are precomputed once per session into a table; and the recipe **name encodes a pipeline** — `A - B - C` means "do A, then B, then C." The Python port keeps both, but replaces the `switch` with a registry.

## Section 2 — The Python concepts you need

### 2.1 — Why normalize (seen, not told)

Two channels at wildly different scales — a linear layer weights every input by the same initial distribution, so the large-scale channel dominates by magnitude alone until training slowly compensates. Per-channel z-scoring removes that accident. Here's the mechanic in plain NumPy (this is exactly what a `Channel - Z-Score` recipe *computes*):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
t = np.arange(200)
ch_small = 0.5 * np.sin(t / 10) + rng.normal(0, 0.1, 200)
ch_large = 800 * np.sin(t / 10) + rng.normal(0, 50, 200)

def zscore(x): return (x - x.mean()) / x.std()

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.4))
a0.plot(t, ch_small, label="channel A (~0.5)"); a0.plot(t, ch_large, label="channel B (~800)")
a0.set_title("raw — channel B dwarfs A"); a0.legend(fontsize=8)
a1.plot(t, zscore(ch_small), label="channel A"); a1.plot(t, zscore(ch_large), label="channel B")
a1.set_title("per-channel z-scored — same scale"); a1.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("after z-scoring, each channel: mean", round(float(zscore(ch_large).mean()), 3),
      " std", round(float(zscore(ch_large).std()), 3))

### 2.2 — The recipe grammar

Recipe names are `Scope - Method [ - Method ...]`, read left to right:

| Token | Meaning |
|---|---|
| **Scope**: `Channel` / `Area` / `Global` | statistics per channel, per probe/area, or over everything |
| **Method**: `Z-Score` | subtract mean, divide by std |
| **Method**: `MinMax - [lo, hi]` | rescale the range to `[lo, hi]` |
| **Modifier**: `Zero Centered` | shift so the midpoint is 0 |
| **Modifier**: `Range 0.5` | scale the half-range to 0.5 |
| `None` | passthrough |

`Channel - Z-Score - Global - MinMax - [-1,1]` means: z-score each channel, THEN globally rescale to [-1, 1]. The registry knows every recipe *name*:

In [ ]:
from neural_data_decoding.data.normalization import list_recipes

print(f"{len(list_recipes())} recipe names registered:")
for r in list_recipes():
    print(" ", r)

### 2.3 — Registered ≠ implemented: reading a mid-migration codebase

Here is a lesson you can only learn from a real port. All 16 names are *registered* — but in the current milestone only **two are actually implemented**: `None` (passthrough) and the full Optimal recipe. The rest are **stubs that raise `NotImplementedError`**, deliberately scheduled for a later milestone (CC). Calling one tells you so, loudly:

In [ ]:
from neural_data_decoding.data.normalization import select_normalization
import pandas as pd

data = np.stack([rng.normal(loc, scale, (50, 1))
                 for loc, scale in [(5, 1), (50, 10), (-20, 4), (100, 25)]])   # (4, 50, 1)

# A stats table (columns Area, Channel, Mean, STD, Min, Max — 1-indexed to match MATLAB):
table = pd.DataFrame({
    "Area": [1, 1, 1, 1], "Channel": [1, 2, 3, 4],
    "Mean": [data[c].mean() for c in range(4)], "STD": [data[c].std() for c in range(4)],
    "Min":  [data[c].min()  for c in range(4)], "Max": [data[c].max() for c in range(4)],
})

try:
    select_normalization(data, table, "Channel - Z-Score")     # a STUB
except NotImplementedError as e:
    print("stub, not yet live:", str(e)[:88], "...")

This is not a bug — it's honest scaffolding. The registry reserves every recipe *name* up front (so configs referencing them parse), while implementations land milestone by milestone. The takeaway for reading any migration: **"the name resolves" and "the code runs" are different questions** — check both. `list_recipes()` shows what's named; only invocation (or the source's stub list) shows what's live.

The two live recipes: `None`, and the production default —

In [ ]:
OPT = "Channel - Z-Score - Global - MinMax - [-1,1] - Zero Centered - Range 0.5"

out = select_normalization(data, table, OPT)          # the ONE fully-implemented composite recipe
print("Optimal recipe WITH a stats table:")
print(f"  input  range [{data.min():7.2f}, {data.max():7.2f}]")
print(f"  output range [{out.min():7.3f}, {out.max():7.3f}]  — transformed, and shape preserved {out.shape}")
print(f"  changed from input? {not np.array_equal(out, data)}")

(The exact output band depends on the *session-level* statistics the table carries; with a real per-session `NormalizationInformation.mat` table the Optimal recipe maps into the network's expected range. The point here is that it *runs and transforms* — the composed multi-step pipeline the name describes.)

### 2.4 — The stats-table requirement and the silent passthrough

The operationally critical fact, and a silent-parity trap: **`select_normalization` needs that statistics table, and without one it returns the data UNCHANGED** — no error, no warning — mirroring `cgg_selectNormalization.m` exactly:

In [ ]:
passthrough = select_normalization(data, None, OPT)     # same recipe, table=None
print("Optimal recipe, but table=None:")
print("  identical to input?", np.array_equal(passthrough, data))   # True — silent no-op!
print("  → forget to build the table and your data flows through UNNORMALIZED, silently.")

The production loader builds the table up front (`cgg_getNormalizationTableFromDataName` → `NormalizationInformation.mat`), so this rarely bites in a full run — but when a normalized model trains worse than expected, **check the table exists before anything else.** A silent no-op is harder to notice than a crash, which is precisely why it deserves a whole section.

## Section 3 — The `neural_data_decoding` implementation

The dispatch — the `switch` replaced by a registry lookup, with the table guard right at the top:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/normalization.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def select_normalization"))
for k in range(i, min(i + 22, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The `None`/empty-table → passthrough branch — §2.4's gotcha, in code, citing the MATLAB source.
* Recipes register themselves into a module dict via a `@register("name")` decorator (scroll up in the file); the stubs are registered the same way via `_make_stub` (scroll down) — so a not-yet-implemented recipe is a *registered* function that raises, not a missing key. Extension without modifying this dispatch — the registry payoff.
* The stats table's `Area`/`Channel` columns are 1-indexed — a MATLAB-parity detail the docstring pins so the indexing code subtracts 1 correctly.

## Section 4 — Hands-on exercises

### Exercise 4.1 — tell named from live

Without running every recipe, write a loop that classifies each name in `list_recipes()` as **live** or **stub**, by catching `NotImplementedError`. (Use the `table` from §2.3; wrap each call in try/except.)

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
live, stub = [], []
for r in list_recipes():
    try:
        select_normalization(data, table, r)
        live.append(r)
    except NotImplementedError:
        stub.append(r)
print(f"LIVE ({len(live)}):", live)
print(f"STUB ({len(stub)}):", len(stub), "recipes pending Milestone CC")

### Exercise 4.2 — catch the silent passthrough

Write the one-line assertion that would have caught §2.4's silent no-op — fail loudly if a recipe returned data identical to its input.

In [ ]:
# Reveal:
out = select_normalization(data, None, OPT)   # table=None → silent passthrough
try:
    assert not np.array_equal(out, data), f"{OPT!r} was a no-op — did you forget the NormalizationTable?"
except AssertionError as e:
    print("caught:", e)

## Section 5 — Common errors

### Normalized data looks unchanged

The empty-table passthrough (§2.4), or you invoked `None`. A table-dependent recipe with no table is a silent no-op. Confirm the `NormalizationTable` exists and is non-empty.

### `NotImplementedError: ... registered but not yet implemented`

You called one of the stub recipes (§2.3). Only `None` and the Optimal recipe are live in the current milestone; the rest arrive in Milestone CC. Use the Optimal recipe, or `None`.

### `KeyError` on the recipe name

Typo — the name isn't even registered. `list_recipes()` is the source of truth; recipe strings are exact, case- and spacing-sensitive.

### `RuntimeWarning: invalid value encountered in divide`

A zero-variance channel (dead/constant) → divide-by-zero in z-scoring → NaN, which flows straight into [02.8](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)'s territory. The stats table's `STD` for that channel is the culprit.

### Python and MATLAB normalization disagree

Usually the 1-indexing of the table's `Area`/`Channel` columns, or a table built from different source data. The dispatch is parity-tested; the table is the usual suspect.

## Section 6 — Further reading

- [scikit-learn preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html) — StandardScaler / MinMaxScaler, the standard-library take on these operations.
- [`src/neural_data_decoding/data/normalization.py`](../../src/neural_data_decoding/data/normalization.py) — the registry, the live Optimal recipe, and the stub scaffold.
- [`tests/unit/test_normalization.py`](../../tests/unit/test_normalization.py) — including the explicit passthrough and stub-raises tests.

Next up: **[03.6 the augmentation per-call contract](03.6_augmentation_per_call_contract.ipynb)** — the last notebook of Module 03, and the subtlest parity trap in the whole data pipeline.